# The Quant Crisis Detector: Three Signals, One System

---

## Why Does This Exist?

Every major market crisis looks different on the surface, but each one leaves fingerprints in three mathematical spaces that most investors never look at:

1. **The texture of volatility** — when markets panic, volatility becomes *rougher*.
   Standard deviation alone misses this.

2. **The latent state of the market** — markets are not random walks. They transition between
   hidden regimes (Bull, Bear, Pre-Crisis) that can be inferred statistically.

3. **The geometry of correlations** — crises don't arrive uniformly. Before they hit,
   the correlation network *changes shape* in measurable ways.

This repository combines these three views into a unified early-warning system.
Each view is independent — they can disagree. When they agree, the signal is strong.

---

## The Three-Signal Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                    MULTI-ASSET RETURN DATA                          │
│                    (yfinance, daily OHLCV)                          │
└──────────┬──────────────────┬───────────────────────┬──────────────┘
           │                  │                        │
           ▼                  ▼                        ▼
┌──────────────────┐ ┌────────────────────┐ ┌─────────────────────┐
│ ROUGH VOLATILITY │ │   HIDDEN MARKOV    │ │  TOPOLOGICAL DATA   │
│     ENGINE       │ │  REGIME DETECTOR   │ │     ANALYSIS        │
│                  │ │                    │ │                     │
│ • DFA Hurst H    │ │ • 5D Gaussian HMM  │ │ • EWMA corr matrix  │
│ • σ_rough vs std │ │ • Baum-Welch EM    │ │ • Mantegna metric   │
│ • Vol surprise   │ │ • Viterbi decoding │ │ • Vietoris-Rips     │
│ • P(regime chg)  │ │ • Hungarian labels │ │ • β₀, β₁ Betti nos. │
└────────┬─────────┘ └────────┬───────────┘ └────────┬────────────┘
         │                    │                       │
         │  feeds features    │                       │
         └──────────────────▶ │                       │
                              │                       │
                    ┌─────────▼───────────────────────▼────┐
                    │         COMBINED SIGNAL              │
                    │                                      │
                    │  P_crisis = 0.30×p_rough             │
                    │           + 0.40×p_regime            │
                    │           + 0.30×p_topology          │
                    │                                      │
                    │  🟢 NORMAL  →  <45%                 │
                    │  🟡 WATCH   →  45–60%               │
                    │  🔶 ELEVATED→  60–75%               │
                    │  ⚠️ CRISIS  →  >75% (2+ signals)    │
                    └──────────────────────────────────────┘
```

---

## The Three Companion Repositories

| Repository | What it studies | Key technique |
|-----------|-----------------|---------------|
| [`rough-volatility-engine`](https://github.com/yourusername/rough-volatility-engine) | Volatility *texture* | Detrended Fluctuation Analysis (DFA) |
| [`market-regime-hmm`](https://github.com/yourusername/market-regime-hmm) | Market *state* | Gaussian HMM + Hungarian assignment |
| [`tda-market-topology`](https://github.com/yourusername/tda-market-topology) | Correlation *geometry* | Vietoris-Rips complex, Betti numbers |

This repository (`quant-crisis-detector`) shows how they interact.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import yfinance as yf
import warnings

from src.crisis_detector import CrisisDetectorPipeline, CombinedSignal
from src.crisis_detector import RoughVolSignal, RegimeSignal, TopologySignal

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 140, 'axes.spines.top': False, 'axes.spines.right': False})

C = {'spy': '#1a3a5c', 'rough': '#e67e22', 'regime': '#2980b9',
     'topo': '#8e44ad', 'combined': '#c0392b', 'alert': '#e74c3c',
     'warn': '#f39c12', 'ok': '#27ae60', 'light_bg': '#f8f9fa'}

print('Three-signal crisis detector loaded.')

## Section 1: What Does Each Signal See?

To understand *how* the three signals differ, we compare them on a **calm period** (2023 Q2)
and a **stressed period** (COVID crash Feb–Mar 2020).

In [ ]:
# Run pipeline on a 3-year window covering multiple regimes
pipeline = CrisisDetectorPipeline(
    tickers=['SPY','QQQ','IWM','GLD','TLT','XLF','XLE','HYG','EEM','XLV']
)
signals = pipeline.run(start='2022-01-01', end='2024-12-31')
spy = yf.download('SPY', start='2022-01-01', end='2024-12-31', progress=False)['Close'].squeeze()

print('Signal summary (2022-2024):')
print(signals[['crisis_prob','hurst_index','stress_score']].describe().round(3))

## Section 2: Signal Correlation Matrix

How correlated are the three signals? High correlation = redundant information.
Low correlation = each signal adds independent value.

In [ ]:
import seaborn as sns

corr_cols = {
    'Rough Vol': 'rough_vol_p',
    'HMM Regime': 'regime_p',
    'Topology': 'topo_p',
    'Combined': 'crisis_prob',
}

corr_df = signals[[v for v in corr_cols.values()]].rename(columns={v:k for k,v in corr_cols.items()})
corr_matrix = corr_df.corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Correlation heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
             vmin=-1, vmax=1, center=0, ax=axes[0],
             linewidths=1, square=True,
             annot_kws={'size': 13, 'weight': 'bold'})
axes[0].set_title('Signal Correlation Matrix (2022–2024)\n'
                   'Low correlation = each signal adds independent information',
                   fontsize=11, fontweight='bold')

# Signal time series overlay
ax = axes[1]
ax.plot(signals.index, signals['rough_vol_p'], color=C['rough'],  lw=1.2, alpha=0.8, label='Rough Volatility')
ax.plot(signals.index, signals['regime_p'],    color=C['regime'], lw=1.2, alpha=0.8, label='HMM Regime')
ax.plot(signals.index, signals['topo_p'],      color=C['topo'],   lw=1.2, alpha=0.8, label='Topology')
ax.plot(signals.index, signals['crisis_prob'], color=C['combined'],lw=2.5,           label='COMBINED')
ax.axhline(0.60, color='gray', lw=0.8, ls='--', alpha=0.5)
ax.set_ylabel('Crisis Probability')
ax.set_ylim(0, 1)
ax.set_title('Individual vs Combined Signal (2022–2024)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
import matplotlib.dates as mdates
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7]))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

plt.tight_layout()
plt.savefig('../data/signal_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelation matrix:')
print(corr_matrix.round(3).to_string())
print('\nKey insight: If correlations are low (<0.5), signals are genuinely independent.')
print('The combined signal captures more information than any individual signal alone.')

## Section 3: Live Snapshot — Today's Readings

In [ ]:
# Get the most recent signal readings
latest = signals.iloc[-1]

rv_sig = RoughVolSignal(
    hurst_index=latest['hurst_index'],
    vol_surprise=latest['vol_surprise'],
    sigma_rough=latest.get('sigma_rough', 0.15),
    sigma_standard=latest.get('sigma_standard', 0.15),
)
re_sig = RegimeSignal(
    regime_label=latest['regime_label'],
    confidence=0.65,
)
tp_sig = TopologySignal(
    betti_0=latest['betti_0'],
    betti_1=latest['betti_1'],
    stress_score=latest['stress_score'],
    stress_level='alert' if latest['stress_score'] > 2.5 else 'elevated' if latest['stress_score'] > 2.0 else 'normal',
)
combined = CombinedSignal(rough_vol=rv_sig, regime=re_sig, topology=tp_sig)

print(f'Most recent signal date: {signals.index[-1].date()}')
print()
print(combined.summary())

## Section 4: How to Use This System

### Decision Framework

| Combined P_crisis | Signals in agreement | Recommended action |
|-------------------|----------------------|--------------------|
| < 45% | 0–1 | Full exposure. Monitor weekly. |
| 45–60% | 1 | Reduce to 75% equity. Increase cash/bonds. |
| 60–75% | 2 | Reduce to 50%. Buy put protection. |
| > 75% | 2–3 | Reduce to 25–0%. Full risk-off. |

### What This System Is NOT

- **Not a timing system**: it doesn't predict *when* the bottom will occur.
- **Not a trading strategy**: it's a risk management overlay, not an alpha signal.
- **Not infallible**: it will generate false positives. The goal is to reduce *severity* of drawdowns, not avoid every correction.

### Academic Value

Each signal in this system corresponds to an active research area:

| Signal | Research Field | Key Papers |
|--------|---------------|------------|
| Rough Volatility | Stochastic Analysis | Gatheral et al. (2018), El Euch & Rosenbaum (2019) |
| HMM Regime | Econometrics | Hamilton (1989), Guidolin & Timmermann (2007) |
| TDA | Applied Topology | Carlsson (2009), Gidea & Katz (2018) |

> Continue to **Notebook 01** (COVID crash) and **Notebook 02** (GFC 2008) for detailed case studies.